## Model From Scratch

In [1]:
import pandas as pd

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

In [2]:
import torch
import torch.nn as nn

class MCQTransformer(nn.Module):

    def __init__(
        self,
        vocab_size,
        hidden_dim=256,
        max_length=256,
        num_heads=8,
        num_layers=4,
        dropout=0.1
    ):
        super().__init__()

        # Word embeddings
        self.embedding = nn.Embedding(
            vocab_size,
            hidden_dim
        )

        # Position embeddings
        self.position_embedding = nn.Embedding(
            max_length,
            hidden_dim
        )

        # One Transformer Encoder layer
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True
        )

        # Stack multiple encoder layers
        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.dropout = nn.Dropout(dropout)

        # Score one option
        self.classifier = nn.Linear(
            hidden_dim,
            1
        )

    def forward(self, input_ids):

        # input shape:
        # (batch_size, 5, sequence_length)

        B, C, L = input_ids.shape

        # Convert into
        # (batch_size*5, sequence_length)

        input_ids = input_ids.view(B * C, L)

        # Position ids
        positions = torch.arange(
            L,
            device=input_ids.device
        ).unsqueeze(0).expand(B * C, L)

        # Embedding
        x = self.embedding(input_ids)

        # Add positional embedding
        x = x + self.position_embedding(positions)

        # Transformer
        x = self.encoder(x)

        # Mean Pooling
        x = x.mean(dim=1)

        x = self.dropout(x)

        # Score
        logits = self.classifier(x)

        # Convert back to
        # (batch_size,5)

        logits = logits.view(B, C)

        return logits

In [3]:
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader

# ---- Tokenizer ----
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

MAX_LEN = 256
OPTION_COLS = ["A", "B", "C", "D", "E"]
LABEL_MAP = {c: i for i, c in enumerate(OPTION_COLS)}


class MCQDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=256, has_labels=True):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.has_labels = has_labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        question = str(row["prompt"])

        input_ids_per_option = []
        for col in OPTION_COLS:
            option_text = str(row[col])
            
            enc = self.tokenizer(
                question,
                option_text,
                truncation=True,
                max_length=self.max_length,
                padding="max_length",
                return_tensors="pt"
            )
            input_ids_per_option.append(enc["input_ids"].squeeze(0))

        # shape: (5, seq_len)
        input_ids = torch.stack(input_ids_per_option, dim=0)

        item = {"input_ids": input_ids}
        if self.has_labels:
            label = LABEL_MAP[str(row["answer"]).strip().upper()]
            item["labels"] = torch.tensor(label, dtype=torch.long)
        return item

train_dataset = MCQDataset(train, tokenizer, max_length=MAX_LEN, has_labels=True)
test_dataset = MCQDataset(test, tokenizer, max_length=MAX_LEN, has_labels=False)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [4]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = MCQTransformer(
    vocab_size=tokenizer.vocab_size,
    hidden_dim=256,
    max_length=256
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=0.01
)

In [5]:
epochs = 5

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for batch in train_loader:

        input_ids = batch["input_ids"].to(device)

        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        logits = model(input_ids)

        loss = criterion(
            logits,
            labels
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}:",
        total_loss / len(train_loader)
    )

Epoch 1: 1.3773447482585908
Epoch 2: 0.7147898621857166
Epoch 3: 0.3074903147257865
Epoch 4: 0.18935406242811587
Epoch 5: 0.08097335909093453


In [6]:
model.eval()

all_logits = []

with torch.no_grad():

    for batch in test_loader:

        input_ids = batch["input_ids"].to(device)

        logits = model(input_ids)

        all_logits.append(
            logits.cpu()
        )

logits = torch.cat(all_logits).numpy()

In [7]:
import numpy as np

label_names = np.array(["A", "B", "C", "D", "E"])
top3 = np.argsort(-logits, axis=1)[:, :3]

predictions = [
    " ".join(label_names[idx])
    for idx in top3
]

submission = test[["id"]].copy()
submission["Prediction"] = predictions
submission.to_csv("submission.csv", index=False)

## Deberta-V3

In [8]:
!pip install -q transformers datasets accelerate wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 86.8 MB/s eta 0:00:00:00:01:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which 

In [9]:
import pandas as pd
import numpy as np
import torch
import wandb
from transformers import AutoTokenizer, AutoModelForMultipleChoice, Trainer, TrainingArguments
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split


MODEL_NAME = "microsoft/deberta-v3-base"
OPTIONS = ['A', 'B', 'C', 'D', 'E']
MAP_LABEL = {opt: idx for idx, opt in enumerate(OPTIONS)}
INV_MAP_LABEL = {idx: opt for idx, opt in enumerate(OPTIONS)}
MAX_LENGTH = 256
LABEL_COL = "answer"   
QUESTION_COL = "prompt" 

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def map3(eval_pred):
    logits, labels = eval_pred
    preds = np.argsort(-logits, axis=1)
    score = 0
    for p, y in zip(preds, labels):
        top3 = p[:3]
        if y == top3[0]:
            score += 1
        elif y == top3[1]:
            score += 0.5
        elif y == top3[2]:
            score += 1/3
    return {"map3": score / len(labels)}


class MCQDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=256, has_labels=True):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.has_labels = has_labels and (LABEL_COL in df.columns)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        question = str(row[QUESTION_COL])
        choices = [str(row[opt]) for opt in OPTIONS]

        encoding = self.tokenizer(
            [question] * 5,
            choices,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        item = {
            "input_ids": encoding["input_ids"],
            "attention_mask": encoding["attention_mask"],
        }

        if self.has_labels:
            label_val = row[LABEL_COL]
            if isinstance(label_val, str):
                label_val = MAP_LABEL[label_val.strip().upper()]
            item["labels"] = torch.tensor(int(label_val), dtype=torch.long)

        return item

train_df, valid_df = train_test_split(
    train,
    test_size=0.2,
    random_state=42
)
train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)

train_dataset = MCQDataset(train_df, tokenizer, max_length=MAX_LENGTH, has_labels=True)
valid_dataset = MCQDataset(valid_df, tokenizer, max_length=MAX_LENGTH, has_labels=True)


wandb.login(key="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")
wandb.init(
    project='24f1002052-t22026',
    name='Distilbert',
    config={
        'model':     MODEL_NAME,
        'epochs':    5,
        'lr':        2e-5,
        'optimizer': 'adamw_torch',
        'max_len':   256
    }
)


model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)

training_args = TrainingArguments(
    output_dir="./outputs",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=15,
    weight_decay=0.02,
    warmup_ratio=0.1,
    max_grad_norm=1.0,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="map3",
    greater_is_better=True,
    optim="adafactor"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    processing_class=tokenizer,
    compute_metrics=map3
)

trainer.train()
wandb.finish()


test_dataset = MCQDataset(test, tokenizer, max_length=MAX_LENGTH, has_labels=False)
pred = trainer.predict(test_dataset)
logits = pred.predictions

labels_arr = np.array(OPTIONS)
order = np.argsort(-logits, axis=1)
predictions = [
    " ".join(labels_arr[idx[:3]])
    for idx in order
]

submission = pd.DataFrame({
    "id": test["id"],
    "Prediction": predictions
})
submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")
print(submission.head())

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.layer.3.output.LayerNorm.weight', 'deberta.encoder.layer.3.output.Laye

Saved submission.csv
   id Prediction
0   1      A D C
1   2      B C D
2   3      B E D
3   4      E C D
4   5      C D A


## DistilBert Model

In [10]:
!pip install -q transformers datasets accelerate wandb

In [11]:
import torch
import wandb
import pandas as pd
import numpy as np

from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    Trainer,
    TrainingArguments
)

In [12]:
import pandas as pd

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

print(train.shape)
train.head()

(2000, 8)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [13]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForMultipleChoice.from_pretrained(
    MODEL_NAME
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForMultipleChoice LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [14]:
from torch.utils.data import Dataset
import torch

class MCQDataset(Dataset):

    def __init__(self, df, tokenizer, max_length=256):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        question = row["prompt"]

        choices = [
            row["A"],
            row["B"],
            row["C"],
            row["D"],
            row["E"],
        ]

        encoding = self.tokenizer(
            [question] * 5,
            choices,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        item = {
            "input_ids": encoding["input_ids"],
            "attention_mask": encoding["attention_mask"],
        }

        if "label" in self.df.columns:
            item["labels"] = torch.tensor(row["label"], dtype=torch.long)

        return item

In [15]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

train["label"] = encoder.fit_transform(train["answer"])

In [16]:
import numpy as np

def map3(eval_pred):

    logits, labels = eval_pred

    preds = np.argsort(-logits, axis=1)

    score = 0

    for p, y in zip(preds, labels):

        top3 = p[:3]

        if y == top3[0]:
            score += 1

        elif y == top3[1]:
            score += 0.5

        elif y == top3[2]:
            score += 1/3

    return {"map3": score / len(labels)}

In [17]:
from sklearn.model_selection import train_test_split

train_df, valid_df = train_test_split(
    train,
    test_size=0.2,
    stratify=train["label"],
    random_state=42
)

train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)

train_dataset = MCQDataset(
    train_df,
    tokenizer,
    max_length=256
)

valid_dataset = MCQDataset(
    valid_df,
    tokenizer,
    max_length=256
)

In [18]:
wandb.login(key="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")
wandb.init(
    project='24f1002052-t22026',
    name='Distilbert',
    config={
        'model':     MODEL_NAME,
        'epochs':    5,
        'lr':        2e-5,
        'optimizer': 'adamw_torch',
        'max_len':   256
    }
)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


In [19]:

training_args = TrainingArguments(

    output_dir="./outputs",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.02,
    warmup_ratio=0.1,
    max_grad_norm=1.0,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="map3",
    greater_is_better=True,
    optim="adamw_torch"
    
)



warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [20]:
trainer = Trainer(

    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    processing_class=tokenizer,
    compute_metrics=map3

)

trainer.train()


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Map3
1,2.965444,2.521842,0.737500
2,1.817936,1.661211,0.907500
3,1.231976,1.008844,0.948333
4,0.953611,0.823896,0.947500
5,0.805795,0.746146,0.955833


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=500, training_loss=1.662206024169922, metrics={'train_runtime': 572.4795, 'train_samples_per_second': 13.974, 'train_steps_per_second': 0.873, 'total_flos': 2649300725760000.0, 'train_loss': 1.662206024169922, 'epoch': 5.0})

In [21]:
test_dataset = MCQDataset(
    test,
    tokenizer,
    max_length=256
)

In [22]:
pred = trainer.predict(test_dataset)

logits = pred.predictions
labels = np.array(["A", "B", "C", "D", "E"])
order = np.argsort(-logits, axis=1)
predictions = [
    " ".join(labels[idx[:3]])
    for idx in order
]

wandb.finish()

In [23]:
print(len(pred.predictions))

500


In [30]:
submission = pd.DataFrame({
    "id": test["id"],
    "Prediction": predictions
})

submission.to_csv("submission.csv", index=False)